# 04 — Inference

Runs the treated-unit inferential battery from [methodology.md §5e](methodology.md):

1. **In-space placebo** — refit treating each donor as treated; compute Brent's permutation p-value
2. **In-time placebo** — refit with $T^{\text{fake}}_0$ = 6 months before real $T_0$
3. **Leave-one-donor-out** — drop each donor with weight > 0.05, recompute the gap, report stability

**Inputs**: requires `02_Fit_Models` to have run (so fits exist in `data/results/`).  
**Outputs**: `data/validation/inference_{test}_{event}_{model}.csv`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from lib.config import T0, T0_FAKE, MODEL_HPARAMS, DONOR_POOL_VARIANT, LOO_MIN_WEIGHT
from lib.data import build_panel, load_fit, save_validation_table
from lib.validation import in_space_placebo, in_time_placebo, leave_one_out, gap_distribution

EVENTS = ['russia', 'hormuz']
WINDOW = 'preferred'
VARIANT = DONOR_POOL_VARIANT
MODELS = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']

print(f'Inferential battery: {EVENTS} × {MODELS}')

Inferential battery: ['russia', 'hormuz'] × ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']
Last run: 2026-06-09 12:06:02


## §5e (i) — In-space placebo

For each model: refit treating each donor as the placebo treated unit; compute the post/pre RMSPE ratio. Brent's rank in the placebo distribution gives the permutation p-value (< 0.10 is the Abadie convention).

*(Note: this is computationally heavy — N_donors × N_models × N_events fits. For convex SCM each placebo takes ~3s; for XGBoost/Bayesian Ridge much faster. Total runtime ~5-10 min.)*

In [2]:
import time
from lib.validation import get_tuned_hparams

iso_results = {}
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS:
        # Use the val-tuned hyperparameters from 02_Fit_Models so placebo runs match the headline fit.
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        if model in ('convex_scm', 'ascm'):
            kwargs['n_random_v'] = 40   # cut down for placebo speed
        t0 = time.time()
        try:
            df = in_space_placebo(model, panel, 'Brent', meta['donors'],
                                  t0=meta['t0'], t_pre_start=meta['t_pre_start'], **kwargs)
            iso_results[(event, model)] = df
            save_validation_table(df, f'inference_inspace_{event}_{model}')
            brent_p = float(df.loc['Brent', 'p_value']) if 'Brent' in df.index else np.nan
            print(f'  {event:6s} / {model:12s}  Brent p = {brent_p:.3f}  ({time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'  {event:6s} / {model:12s}  ERROR: {str(e)[:60]}')

  russia / convex_scm    Brent p = 0.500  (11s)


  russia / ascm          Brent p = 0.600  (11s)
  russia / elastic_net   Brent p = 0.400  (0s)


  russia / xgboost       Brent p = 0.150  (5s)
  russia / bayesian_ridge  Brent p = 0.600  (0s)


  hormuz / convex_scm    Brent p = 0.050  (12s)


  hormuz / ascm          Brent p = 0.050  (12s)
  hormuz / elastic_net   Brent p = 0.050  (0s)


  hormuz / xgboost       Brent p = 0.050  (5s)
  hormuz / bayesian_ridge  Brent p = 0.050  (0s)
Last run: 2026-06-09 12:06:59


In [3]:
# Summary: Brent's p-value across models × events
rows = []
for (event, model), df in iso_results.items():
    if 'Brent' not in df.index:
        continue
    row = df.loc['Brent'].to_dict()
    row.update({'event': event, 'model': model})
    rows.append(row)

iso_brent = pd.DataFrame(rows)
if len(iso_brent):
    save_validation_table(iso_brent, 'inference_inspace_brent_summary')
    iso_brent.round(4)
else:
    print('No in-space placebo results to summarize.')

Last run: 2026-06-09 12:06:59


## §5e (ii) — In-time placebo

Refit with fake $T_0$ = 6 months before the real event. The gap in the fake post-period (from $T^{\text{fake}}_0$ to real $T_0$) should be small — large gap means the SCM is finding spurious effects.

In [4]:
from lib.validation import get_tuned_hparams

intime_rows = []
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    t0_fake = T0_FAKE[event]
    for model in MODELS:
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        try:
            r = in_time_placebo(model, panel, 'Brent', meta['donors'],
                                t0_fake=t0_fake, t_pre_start=meta['t_pre_start'], **kwargs)
            # Compute gap in the fake post-period (t0_fake to real t0)
            fake_post = r['gap'][(r['gap'].index >= t0_fake) & (r['gap'].index < meta['t0'])]
            mean_fake_gap_pct = float(100 * (np.exp(fake_post.mean()) - 1)) if len(fake_post) > 0 else np.nan
            intime_rows.append({
                'event': event, 'model': model,
                't0_fake': str(t0_fake.date()),
                'fake_post_obs': len(fake_post),
                'mean_fake_gap_pct': mean_fake_gap_pct,
                'pre_rmspe_log': r['rmspe_pre'],
            })
        except Exception as e:
            intime_rows.append({'event': event, 'model': model, 'error': str(e)[:60]})

intime_df = pd.DataFrame(intime_rows)
save_validation_table(intime_df, 'inference_intime')
intime_df.round(4)

,event,model,t0_fake,fake_post_obs,mean_fake_gap_pct,pre_rmspe_log
0,russia,convex_scm,2021-08-24,129,5.7452,0.1209
1,russia,ascm,2021-08-24,129,3.4461,0.1209
2,russia,elastic_net,2021-08-24,129,2.7297,0.0538
3,russia,xgboost,2021-08-24,129,16.7375,0.0392
4,russia,bayesian_ridge,2021-08-24,129,-2.8132,0.0337
5,hormuz,convex_scm,2025-08-28,129,-10.2182,0.0591
6,hormuz,ascm,2025-08-28,129,-7.5071,0.0591
7,hormuz,elastic_net,2025-08-28,129,-3.8030,0.0548
8,hormuz,xgboost,2025-08-28,129,-6.7316,0.0461
9,hormuz,bayesian_ridge,2025-08-28,129,-1.8663,0.0359


Last run: 2026-06-09 12:07:04


## §5e (iii) — Leave-one-donor-out

Drop each high-weight donor and recompute the gap. Stability (small range around the baseline gap) means no single donor is driving the headline.

In [5]:
from lib.validation import get_tuned_hparams

for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS:
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        if model in ('convex_scm', 'ascm'):
            kwargs['n_random_v'] = 40
        try:
            loo_results = leave_one_out(model, panel, 'Brent', meta['donors'],
                                        t0=meta['t0'], t_pre_start=meta['t_pre_start'],
                                        min_weight=LOO_MIN_WEIGHT, **kwargs)
            dist = gap_distribution(loo_results, t0=meta['t0'])
            save_validation_table(dist, f'inference_loo_{event}_{model}')
            print(f'\n{event} / {model} (baseline + {len(dist)-1} leave-outs):')
            print(dist.round(3).to_string())
        except Exception as e:
            print(f'{event} / {model}  ERROR: {str(e)[:80]}')


russia / convex_scm (baseline + 2 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        33.686          37.599       59.310        3.432
Coffee           59.555          60.564       96.784       29.468
Sugar            32.810          37.013       58.340        3.734



russia / ascm (baseline + 16 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         10.874          15.318       49.745      -25.439
Silver             9.364          13.476       50.862      -28.848
Platinum          -0.141           3.102       57.163      -36.200
Gold               9.159          13.244       47.157      -23.853
Coffee             9.846          13.383       45.799      -25.780
Sugar             13.969          18.330       52.801      -20.962
LiveCattle        10.341          14.505       50.418      -25.907
SP500             12.208          16.560       49.748      -23.613
Nikkei             9.999          14.147       51.248      -26.745
JPY               15.464          19.985       50.071      -20.438
CHF               10.766          15.098       49.916      -25.786
KRW               13.243          17.261       51.158      -20.886
ZAR                


russia / xgboost (baseline + 4 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         29.901          27.621       79.258        2.916
Sugar             35.291          31.602       97.976        7.002
LiveCattle        31.148          29.006       84.533        3.498
SP500             24.873          25.901       50.085       -3.944
TLT               35.035          30.427      106.732        4.329

russia / bayesian_ridge (baseline + 19 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         -8.989          -4.380       41.619      -51.410
Silver           -11.319          -5.471       42.416      -54.744
Platinum         -21.871         -17.052       44.246      -62.387
Gold              -6.103          -1.586       36.799      -43.136
Coffee           -13.071       


hormuz / convex_scm (baseline + 3 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        58.518          59.408      101.476       14.345
Sugar            54.140          58.436       95.610        7.985
JPY              62.369          63.819      106.728       16.252
TLT              57.111          58.359       99.483       14.704



hormuz / ascm (baseline + 4 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        60.418          61.735      104.230       16.658
Coffee           63.973          65.123      109.727       19.593
Sugar            62.406          66.628      106.243       16.011
JPY              62.452          64.103      108.158       17.129
TLT              58.310          59.915      101.013       16.519

hormuz / elastic_net (baseline + 2 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        60.657          63.660      104.647       16.613
Gold             59.388          63.036      103.156       15.166
Coffee           66.049          68.755      112.829       21.641



hormuz / xgboost (baseline + 6 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         55.561          59.249       98.198       11.375
Silver            55.295          58.966       97.845       11.210
Gold              55.207          58.880       97.738       11.138
Coffee            60.452          64.160      104.310       14.842
LiveCattle        55.507          59.183       98.115       11.359
CHF               55.385          59.067       97.970       11.256
INR               55.365          59.048       97.947       11.237

hormuz / bayesian_ridge (baseline + 15 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         46.458          50.138       83.514       12.349
Silver            47.091          50.550       84.541       12.877
Platinum          46.441       